<a href="https://colab.research.google.com/github/francescopassante/GETMeshClassificator/blob/main/GET/src/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%cd /content

/content


In [2]:
%rm -rf GETMeshClassificator/
!git clone https://github.com/francescopassante/GETMeshClassificator.git
%cd /content

Cloning into 'GETMeshClassificator'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 316 (delta 96), reused 264 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (316/316), 289.15 KiB | 13.14 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/GETMeshClassificator/GET/src


In [4]:
# Download dataset with gdown:
!gdown 1stCA7v4wVr8NRfVEetXKRRFVZbnWmf5U
!unzip -q SHREC11.zip

Downloading...
From (original): https://drive.google.com/uc?id=1stCA7v4wVr8NRfVEetXKRRFVZbnWmf5U
From (redirected): https://drive.google.com/uc?id=1stCA7v4wVr8NRfVEetXKRRFVZbnWmf5U&confirm=t&uuid=c5a686b2-bdd8-454c-8d95-81eb416adf86
To: /content/SHREC11.zip
100% 701M/701M [00:21<00:00, 32.1MB/s]


In [5]:
%cd GETMeshClassificator/GET/src

/content/GETMeshClassificator/GET/src


In [33]:
import GET
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

train_loader, test_loader = GET.load_data(
        mesh_directory="../../../SHREC11/processed/",
        labels_file="../../../SHREC11/classes.txt",
        N=9,
        train_percent=0.02,
)
print(len(train_loader), len(test_loader))

11 588


In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GET.GETClassifier(N=9, channels=12, heads = 2, out_classes=30).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.3)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model has {num_params} parameters")

Model has 9114 parameters


In [39]:
def train(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=1,
    accumulation_steps=16,
):
    model.train()
    loss_hist = []

    for epoch in range(epochs):
        total_loss = 0.0

        # Start with zeroed gradients for the first accumulation block
        optimizer.zero_grad()

        for i, mesh in enumerate(tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")):
            x = mesh["x"].to(device).squeeze(0)
            neighbors = mesh["neighbors"].to(device).squeeze(0)
            mask = mesh["mask"].to(device).squeeze(0)
            parallel_transport_matrices = (
                mesh["parallel_transport_matrices"].to(device).squeeze(0)
            )
            rel_pos_u = mesh["rel_pos"].to(device).squeeze(0)
            labels = mesh["label"].to(device).long().squeeze(0)

            # Forward
            outputs = model(x, neighbors, mask, parallel_transport_matrices, rel_pos_u)
            # Print the index of dimension with max output
            # print("mesh file: ", mesh["filenumber"])
            # print("predicted label: ", torch.argmax(outputs).item())
            # print("True label: ", labels.item())
            raw_loss = criterion(outputs.unsqueeze(0), labels.unsqueeze(0))

            if torch.isnan(raw_loss):
                print("NAN LOSS")

            # Scale the loss for gradient accumulation
            scaled_loss = raw_loss / accumulation_steps
            scaled_loss.backward()

            # Accumulate the raw (unscaled) loss for logging
            total_loss += raw_loss.item()

            # Step and zero gradients at the accumulation boundary (or final batch)
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
                #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

        # Average epoch loss (uses unscaled batch losses)
        epoch_loss = total_loss / len(dataloader)
        print("epoch_loss: ", epoch_loss)
        loss_hist.append(epoch_loss)
        scheduler.step()
        if epoch % 10 == 0 or epoch == epochs - 1:
            # Save checkpoint with meaningful loss value (epoch average)
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": epoch_loss,
            }
            save_path = "checkpoint.pth"
            torch.save(checkpoint, save_path)
            # print(f"Saved checkpoint to {save_path} (epoch_loss={epoch_loss:.6f})")

    return loss_hist

In [40]:
train(model,
    train_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=20,
    accumulation_steps=4)

Epoch 1/20: 100%|██████████| 11/11 [00:02<00:00,  4.59it/s]


epoch_loss:  3.2027890248732134


Epoch 2/20: 100%|██████████| 11/11 [00:02<00:00,  4.90it/s]


epoch_loss:  2.878957444971258


Epoch 3/20: 100%|██████████| 11/11 [00:02<00:00,  4.79it/s]


...

Epoch 19/20: 100%|██████████| 11/11 [00:02<00:00,  4.69it/s]


epoch_loss:  1.7977844585071912


Epoch 20/20: 100%|██████████| 11/11 [00:02<00:00,  4.43it/s]

epoch_loss:  1.8101200949062


[3.2027890248732134,
 2.878957444971258,
 2.7211144837466152,
 2.602943116968328,
 2.488415999846025,
 2.3727424144744873,
 2.302488337863575,
 2.2637449394572866,
 2.2097168293866245,
 2.1590526537461714,
 2.1336384253068403,
 2.0977884097532793,
 2.0525956587357954,
 2.0124486576427114,
 1.9938798275860874,
 1.9417829730293967,
 1.9097078009085222,
 1.8606543378396467,
 1.7977844585071912,
 1.8101200949062]